# AndinaLog 03B | Warehouse Costs | Tratamiento v2

Se conserva la fila completa de Tarija y se excluye su copia incompleta posterior. No se imputa rotación.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ENTORNO='auto';RUTA_PROYECTO_DRIVE='/content/drive/MyDrive/GIAD'
COLUMNAS_BRONZE=['centro_distribucion','rotacion_stock_dias','perdida_mermas_bob','periodo_mes','costo_almacenamiento_mensual_bob']
CENTROS={'Cochabamba','La Paz','Santa Cruz','Oruro','Tarija'}
def encontrar_raiz():
    if ENTORNO=='drive' or (ENTORNO=='auto' and 'google.colab' in sys.modules):
        from google.colab import drive;drive.mount('/content/drive');return Path(RUTA_PROYECTO_DRIVE)
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/'datasets/AndinaLog_03B_Bronce/andinalog_warehouse_costs.csv').is_file():return p
    raise FileNotFoundError('No se encontró la raíz')
RAIZ=encontrar_raiz();RUTA_BRONZE=RAIZ/'datasets/AndinaLog_03B_Bronce/andinalog_warehouse_costs.csv'
ENTRADA=RAIZ/'proyecto-integrador/01_diagnostico/andinalog_warehouse_costs/salidas/andinalog_warehouse_costs_didactico_v2_diagnosticado.csv';SALIDAS=RAIZ/'proyecto-integrador/02_tratamiento/andinalog_warehouse_costs/salidas'
df=pd.read_csv(ENTRADA,dtype='string',encoding='utf-8-sig',keep_default_na=False);bronze=pd.read_csv(RUTA_BRONZE,dtype='string',encoding='utf-8-sig',keep_default_na=False);pd.testing.assert_frame_equal(df[COLUMNAS_BRONZE],bronze);original=df.copy();df=df.rename(columns={'en_cuarentena':'en_cuarentena_diagnostico'});print('Entrada:',len(df))


## Valores tratados y decisión final

Completar una copia que luego debe excluirse no agrega una observación válida al Silver.


In [ ]:
df['acciones_tratamiento']='';df['motivos_tratamiento']='';df['centro_distribucion_tratado']=df.centro_distribucion.str.strip();df['periodo_mes_tratado']=df.periodo_mes.str.strip()
for x in ['rotacion_stock_dias','perdida_mermas_bob','costo_almacenamiento_mensual_bob']:df[f'{x}_tratado']=pd.to_numeric(df[x].str.strip(),errors='coerce')
clave=df.centro_distribucion_tratado+'|'+df.periodo_mes_tratado;copia=clave.duplicated()&df.rotacion_stock_dias.str.strip().eq('');df['rotacion_stock_dias_imputada']=False;df['motivo_cuarentena_final']=''
def q(m,mot):
 m=pd.Series(m,index=df.index).fillna(False).astype(bool);p=df.loc[m,'motivo_cuarentena_final'];df.loc[m,'motivo_cuarentena_final']=p.where(p.eq(''),p+' | ')+mot
q(copia,'Copia incompleta posterior de la misma clave centro-periodo');q(~df.centro_distribucion_tratado.isin(CENTROS),'Centro inválido');q(pd.to_datetime(df.periodo_mes_tratado,format='%Y-%m',errors='coerce').isna(),'Periodo inválido');q(df.rotacion_stock_dias_tratado.isna()&~copia,'Rotación faltante o no numérica');q(df.rotacion_stock_dias_tratado.notna()&df.rotacion_stock_dias_tratado.le(0),'Rotación no positiva')
for x in ['perdida_mermas_bob','costo_almacenamiento_mensual_bob']:q(df[f'{x}_tratado'].isna()|df[f'{x}_tratado'].lt(0),f'{x} inválido')
df['en_cuarentena_final']=df.motivo_cuarentena_final.ne('');df['decision_tratamiento']=np.where(df.en_cuarentena_final,'CUARENTENA','SILVER');silver=df.loc[~df.en_cuarentena_final].copy();cuarentena=df.loc[df.en_cuarentena_final].copy()


## Comprobaciones y exportación

Silver contiene una fila completa por centro para agosto de 2026.


In [ ]:
assert len(df)==len(silver)+len(cuarentena)==6;assert (len(silver),len(cuarentena))==(5,1);assert not silver.duplicated(['centro_distribucion_tratado','periodo_mes_tratado']).any();assert silver.rotacion_stock_dias_tratado.notna().all();assert not df.rotacion_stock_dias_imputada.any();pd.testing.assert_frame_equal(df[[x for x in original.columns if x!='en_cuarentena']],original[[x for x in original.columns if x!='en_cuarentena']]);metricas={'filas_entrada':6,'filas_silver':5,'filas_cuarentena_final':1,'filas_recuperadas':0,'rotaciones_imputadas':0,'centros_silver':5,'periodos_silver':1};reporte=pd.DataFrame([{'metrica':k,'valor':v} for k,v in metricas.items()]);SALIDAS.mkdir(parents=True,exist_ok=True);b='andinalog_warehouse_costs_didactico_v2_';silver.to_csv(SALIDAS/(b+'silver.csv'),index=False,encoding='utf-8-sig');cuarentena.to_csv(SALIDAS/(b+'cuarentena_final.csv'),index=False,encoding='utf-8-sig');reporte.to_csv(SALIDAS/(b+'reporte_calidad.csv'),index=False,encoding='utf-8-sig');print(metricas);display(df)
